In [1]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

# Load the combined CSV
comparison = pd.read_csv("comparison_multiseed_summary_all_perms.csv")

# Pick the metric you want to test
metric = "accuracy"   # try also: "f1_macro", "f1_micro", "auc_macro", "ap_micro"

# Pivot to wide format: one row per seed, one column per permutation
wide = (
    comparison.pivot_table(index="seed", columns="perm", values=metric)
    .reset_index()
    .sort_values("seed")
    .reset_index(drop=True)
)

print(wide.head())
print("\nGroups:", list(wide.columns[1:]))

# Pairwise comparisons
groups = [c for c in wide.columns if c != "seed"]
pairs = list(combinations(groups, 2))

results = []

for a, b in pairs:
    x = wide[a].astype(float).to_numpy()
    y = wide[b].astype(float).to_numpy()

    # Keep only paired rows where both values exist
    valid = ~np.isnan(x) & ~np.isnan(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 2:
        continue

    # Paired non-parametric test
    stat, p = wilcoxon(x, y, alternative="two-sided", zero_method="wilcox")

    results.append({
        "group_a": a,
        "group_b": b,
        "n_pairs": len(x),
        "mean_a": float(np.mean(x)),
        "mean_b": float(np.mean(y)),
        "diff_mean": float(np.mean(y) - np.mean(x)),
        "stat": float(stat),
        "p_value": float(p),
    })

results_df = pd.DataFrame(results)

if not results_df.empty:
    results_df["p_adjust_bonf"] = multipletests(
        results_df["p_value"], method="bonferroni"
    )[1]
    results_df["p_adjust_fdr"] = multipletests(
        results_df["p_value"], method="fdr_bh"
    )[1]

results_df = results_df.sort_values("p_value")
display(results_df)

perm  seed      perm0      perm1      perm2      perm3      perm4
0       40  98.406170  98.457584  97.892031  98.303342  98.508997
1       41  98.560411  98.303342  98.560411  97.840617  98.508997
2       42  98.714653  98.303342  98.406170  98.508997  98.560411
3       43  98.714653  98.200514  98.200514  98.354756  98.611825
4       44  98.508997  98.251928  97.892031  98.457584  98.560411

Groups: ['perm0', 'perm1', 'perm2', 'perm3', 'perm4']


,group_a,group_b,n_pairs,mean_a,mean_b,diff_mean,stat,p_value,p_adjust_bonf,p_adjust_fdr
2,perm0,perm3,5,98.580977,98.293059,-0.287918,0.0,0.0625,0.625,0.208333
6,perm1,perm4,5,98.303342,98.550129,0.246787,0.0,0.0625,0.625,0.208333
9,perm3,perm4,5,98.293059,98.550129,0.257069,0.0,0.0625,0.625,0.208333
1,perm0,perm2,5,98.580977,98.190231,-0.390746,0.0,0.1250,1.000,0.208333
8,perm2,perm4,5,98.190231,98.550129,0.359897,1.0,0.1250,1.000,0.208333
0,perm0,perm1,5,98.580977,98.303342,-0.277635,1.0,0.1250,1.000,0.208333
7,perm2,perm3,5,98.190231,98.293059,0.102828,5.0,0.6250,1.000,0.763889
4,perm1,perm2,5,98.303342,98.190231,-0.113111,3.0,0.6250,1.000,0.763889
3,perm0,perm4,5,98.580977,98.550129,-0.030848,5.5,0.6875,1.000,0.763889
5,perm1,perm3,5,98.303342,98.293059,-0.010283,7.0,1.0000,1.000,1.000000


In [2]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

# Load the combined CSV
comparison = pd.read_csv("comparison_multiseed_summary_all_perms.csv")

# Choose the metric
metric = "accuracy"   # try: "f1_macro", "f1_micro", "auc_macro", "ap_micro"

# Keep only rows with numeric seed values
comparison["seed"] = pd.to_numeric(comparison["seed"], errors="coerce")
comparison = comparison.dropna(subset=["seed"]).copy()
comparison["seed"] = comparison["seed"].astype(int)

# Pivot to wide format: one row per seed, one column per permutation
wide = (
    comparison.pivot_table(index="seed", columns="perm", values=metric)
    .reset_index()
    .sort_values("seed")
    .reset_index(drop=True)
)

print("Available groups:")
print(list(wide.columns[1:]))

# Pairwise comparisons
groups = [c for c in wide.columns if c != "seed"]
pairs = list(combinations(groups, 2))

results = []

for a, b in pairs:
    x = wide[a].astype(float).to_numpy()
    y = wide[b].astype(float).to_numpy()

    # Keep only paired rows where both values exist
    valid = ~np.isnan(x) & ~np.isnan(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 2:
        continue

    stat, p = wilcoxon(x, y, alternative="two-sided", zero_method="wilcox")

    results.append({
        "group_a": a,
        "group_b": b,
        "n_pairs": len(x),
        "mean_a": float(np.mean(x)),
        "mean_b": float(np.mean(y)),
        "diff_mean": float(np.mean(y) - np.mean(x)),
        "stat": float(stat),
        "p_value": float(p),
    })

results_df = pd.DataFrame(results)

if not results_df.empty:
    results_df["p_adjust_bonf"] = multipletests(
        results_df["p_value"], method="bonferroni"
    )[1]
    results_df["p_adjust_fdr"] = multipletests(
        results_df["p_value"], method="fdr_bh"
    )[1]

results_df = results_df.sort_values("p_value")
display(results_df)

Available groups:
['perm0', 'perm1', 'perm2', 'perm3', 'perm4']


,group_a,group_b,n_pairs,mean_a,mean_b,diff_mean,stat,p_value,p_adjust_bonf,p_adjust_fdr
2,perm0,perm3,5,98.580977,98.293059,-0.287918,0.0,0.0625,0.625,0.208333
6,perm1,perm4,5,98.303342,98.550129,0.246787,0.0,0.0625,0.625,0.208333
9,perm3,perm4,5,98.293059,98.550129,0.257069,0.0,0.0625,0.625,0.208333
1,perm0,perm2,5,98.580977,98.190231,-0.390746,0.0,0.1250,1.000,0.208333
8,perm2,perm4,5,98.190231,98.550129,0.359897,1.0,0.1250,1.000,0.208333
0,perm0,perm1,5,98.580977,98.303342,-0.277635,1.0,0.1250,1.000,0.208333
7,perm2,perm3,5,98.190231,98.293059,0.102828,5.0,0.6250,1.000,0.763889
4,perm1,perm2,5,98.303342,98.190231,-0.113111,3.0,0.6250,1.000,0.763889
3,perm0,perm4,5,98.580977,98.550129,-0.030848,5.5,0.6875,1.000,0.763889
5,perm1,perm3,5,98.303342,98.293059,-0.010283,7.0,1.0000,1.000,1.000000


In [3]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import wilcoxon

# -----------------------------
# Config
# -----------------------------
metric = "accuracy"   # try: "f1_macro", "f1_micro", "auc_macro", "ap_micro"

# -----------------------------
# Load + clean
# -----------------------------
comparison = pd.read_csv("comparison_multiseed_summary_all_perms.csv")
comparison["seed"] = pd.to_numeric(comparison["seed"], errors="coerce")
comparison = comparison.dropna(subset=["seed"])

wide = comparison.pivot_table(index="seed", columns="perm", values=metric).reset_index()
groups = [c for c in wide.columns if c != "seed"]

# -----------------------------
# Pairwise tests
# -----------------------------
rows = []
for a, b in combinations(groups, 2):
    x, y = wide[a].to_numpy(float), wide[b].to_numpy(float)
    valid = ~np.isnan(x) & ~np.isnan(y)
    x, y = x[valid], y[valid]
    if len(x) < 2:
        continue

    _, p = wilcoxon(x, y, method="exact")
    rows.append({
        "group_a": a,
        "group_b": b,
        "mean_a": round(x.mean(), 4),
        "mean_b": round(y.mean(), 4),
        "p_value": round(p, 4),
    })

results = pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)

# -----------------------------
# Holm-Bonferroni correction (self-contained, no statsmodels)
# -----------------------------
m = len(results)
p_sorted = results["p_value"].to_numpy()  # already sorted ascending
adj = np.empty(m)
prev = 0
for i in range(m):
    val = min(max((m - i) * p_sorted[i], prev), 1.0)
    adj[i] = val
    prev = val
results["p_holm"] = adj.round(4)

print(results.to_string(index=False))
results.to_csv("pairwise_wilcoxon_summary.csv", index=False)

group_a group_b  mean_a  mean_b  p_value  p_holm
  perm0   perm3 98.5810 98.2931   0.0625   0.625
  perm1   perm4 98.3033 98.5501   0.0625   0.625
  perm3   perm4 98.2931 98.5501   0.0625   0.625
  perm0   perm2 98.5810 98.1902   0.1250   0.875
  perm2   perm4 98.1902 98.5501   0.1250   0.875
  perm0   perm1 98.5810 98.3033   0.1250   0.875
  perm2   perm3 98.1902 98.2931   0.6250   1.000
  perm1   perm2 98.3033 98.1902   0.6250   1.000
  perm0   perm4 98.5810 98.5501   0.8125   1.000
  perm1   perm3 98.3033 98.2931   1.0000   1.000


# 12-lead ResNet vs 1-lead ResNet (Data Type 1)

#

## 

In [13]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon


def clean_seeds(path):
    """Load CSV and remove rows with invalid seeds."""
    df = pd.read_csv(path)
    df["seed"] = pd.to_numeric(df["seed"], errors="coerce")
    return df.dropna(subset=["seed"])


def wilcoxon_compare(
    path_a,
    path_b,
    label_a,
    label_b,
    metrics,
    output_csv="wilcoxon_summary.csv",
):
    """
    Perform paired Wilcoxon signed-rank tests between two experiment summaries.

    Parameters
    ----------
    path_a : str
        CSV path for experiment A.
    path_b : str
        CSV path for experiment B.
    label_a : str
        Display name for experiment A.
    label_b : str
        Display name for experiment B.
    metrics : list[str]
        Metrics to compare.
    output_csv : str
        Output CSV filename.

    Returns
    -------
    pandas.DataFrame
        Table containing means, p-values, and Holm-corrected p-values.
    """

    df_a = clean_seeds(path_a)
    df_b = clean_seeds(path_b)

    rows = []

    for metric in metrics:
        a = df_a[["seed", metric]].rename(columns={metric: "a"})
        b = df_b[["seed", metric]].rename(columns={metric: "b"})
        paired = a.merge(b, on="seed")

        x = paired["a"].astype(float).to_numpy()
        y = paired["b"].astype(float).to_numpy()

        _, p = wilcoxon(x, y, method="exact")

        rows.append({
            "metric": metric,
            f"mean_{label_a}": round(x.mean(), 4),
            f"mean_{label_b}": round(y.mean(), 4),
            "p_value": p,
        })

    results = pd.DataFrame(rows)

    # Holm-Bonferroni correction
    m = len(results)
    order = np.argsort(results["p_value"].to_numpy())
    adj = np.empty(m)
    prev = 0

    for rank, idx in enumerate(order):
        val = min(max((m - rank) * results.loc[idx, "p_value"], prev), 1.0)
        adj[idx] = val
        prev = val

    results["p_holm"] = adj

    # Round numeric columns for display
    numeric_cols = results.select_dtypes(include=np.number).columns
    results[numeric_cols] = results[numeric_cols].round(4)

    print(results.to_string(index=False))


In [14]:
metrics = ["accuracy", "f1_macro", "f1_micro", "auc_macro", "ap_micro"]

wilcoxon_compare(
    path_a="EXPERIMENT_2d_leg_typ1_smallcnn_randomseed/small_cnn_lr1e-3_multiseed_summary.csv",
    path_b="EXPERIMENT_2d_resnet_typ1_ECGresnet_randomseed/2d_resnet_ECGresnetlr1e-3_seed44_multiseed_summary.csv",
    label_a="OPI (typ1 - Legendre 100×100)",
    label_b="typ1 - Pretrained 2D ResNet",
    metrics=metrics,
    output_csv="wilcoxon_summary.csv",
)

   metric  mean_OPI (typ1 - Legendre 100×100)  mean_typ1 - Pretrained 2D ResNet  p_value  p_holm
 accuracy                               98.58                             98.09   0.0625  0.3125
 f1_macro                              0.9853                            0.9805   0.0625  0.3125
 f1_micro                              0.9858                            0.9809   0.0625  0.3125
auc_macro                              0.9992                            0.9986   0.0625  0.3125
 ap_micro                              0.9989                             0.998   0.0625  0.3125


In [15]:
path_a = "EXPERIMENT_resnet_1d_selfeeg_typ1_randomseed/resnet_1d_selfeeg_lr1e-3_seed44_multiseed_summary.csv"

path_b = "EXPERIMENT_resnet_1d_selfeeg_typ1_singlelead1_randomseed/resnet_1d_selfeeg_lr1e-3_seed44_multiseed_summary.csv"


wilcoxon_compare(
    path_a=path_a,
    path_b=path_b,
    label_a="typ1 - Pretrained 1D ResNet (all leads)",
    label_b="typ1 - Pretrained 1D ResNet (single lead)",
    metrics=metrics,
)

   metric  mean_typ1 - Pretrained 1D ResNet (all leads)  mean_typ1 - Pretrained 1D ResNet (single lead)  p_value  p_holm
 accuracy                                          98.4                                           98.03   0.3125   0.625
 f1_macro                                        0.9833                                          0.9799   0.1875  0.5625
 f1_micro                                         0.984                                          0.9803   0.3125   0.625
auc_macro                                        0.9989                                          0.9981    0.125     0.5
 ap_micro                                        0.9985                                          0.9972   0.0625  0.3125


In [17]:
path_a = "EXPERIMENT_2d_leg_typ1_smallcnn_randomseed/small_cnn_lr1e-3_multiseed_summary.csv"

path_b = "EXPERIMENT_2d_resnet_typ1_ECGresnet_randomseed/2d_resnet_ECGresnetlr1e-3_seed44_multiseed_summary.csv"

wilcoxon_compare(
    path_a=path_a,
    path_b=path_b,
    label_a="typ1 - OPI (Legendre 100×100)",
    label_b="typ1 - Pretrained 2D ResNet",
    metrics=metrics,
)

   metric  mean_typ1 - OPI (Legendre 100×100)  mean_typ1 - Pretrained 2D ResNet  p_value  p_holm
 accuracy                               98.58                             98.09   0.0625  0.3125
 f1_macro                              0.9853                            0.9805   0.0625  0.3125
 f1_micro                              0.9858                            0.9809   0.0625  0.3125
auc_macro                              0.9992                            0.9986   0.0625  0.3125
 ap_micro                              0.9989                             0.998   0.0625  0.3125


In [20]:
path_a = "EXPERIMENT_2d_leg_typ1_smallcnn_randomseed/small_cnn_lr1e-3_multiseed_summary.csv"

path_b = "EXPERIMENT_2d_resnet_typ1_ECGresnet_pretrain_FALSE_randomseed/2d_resnet_ECGresnetlr1e-3_pretrain_FALSE_seed44_multiseed_summary.csv"

wilcoxon_compare(
    path_a=path_a,
    path_b=path_b,
    label_a="typ1 - OPI (Legendre 100×100)",
    label_b="typ1 - non-Pretrained 2D ResNet ",
    metrics=metrics,
)

   metric  mean_typ1 - OPI (Legendre 100×100)  mean_typ1 - non-Pretrained 2D ResNet   p_value  p_holm
 accuracy                               98.58                                  98.23   0.0625  0.3125
 f1_macro                              0.9853                                 0.9819   0.0625  0.3125
 f1_micro                              0.9858                                 0.9823   0.0625  0.3125
auc_macro                              0.9992                                 0.9988   0.0625  0.3125
 ap_micro                              0.9989                                 0.9983   0.0625  0.3125


In [21]:
path_a = "EXPERIMENT_2d_leg_typ1_smallcnn_randomseed/small_cnn_lr1e-3_multiseed_summary.csv"



path_b = "EXPERIMENT_resnet_1d_selfeeg_typ1_randomseed/resnet_1d_selfeeg_lr1e-3_seed44_multiseed_summary.csv"



wilcoxon_compare(
    path_a=path_a,
    path_b=path_b,
    label_a="typ1 - OPI (Legendre 100×100)",
    label_b="typ1 -  1d ResNet (all leads)",
    metrics=metrics,
)


   metric  mean_typ1 - OPI (Legendre 100×100)  mean_typ1 -  1d ResNet (all leads)  p_value  p_holm
 accuracy                               98.58                                98.4     0.25  0.5625
 f1_macro                              0.9853                              0.9833   0.1875  0.5625
 f1_micro                              0.9858                               0.984     0.25  0.5625
auc_macro                              0.9992                              0.9989   0.0625  0.3125
 ap_micro                              0.9989                              0.9985   0.0625  0.3125
